# T20 Analytics — Batting, Bowling and Match Phases

**Goal:** from ball-by-ball rows to three defensible player leaderboards and a
phase analysis that explains *how* T20 matches are won.

**Data:** `data/t20_matches.csv` (summary, 2 rows per match) and
`data/t20_deliveries.csv` (ball-by-ball for 3 matches).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (11, 4)

matches = pd.read_csv("data/t20_matches.csv")
balls = pd.read_csv("data/t20_deliveries.csv")
print(matches.shape, balls.shape)
matches.head(4)

## 1 · Validation first
Cricket data has hidden traps: wides that don't count as legal balls,
"runs" that include extras, and innings 2 totals that stop when the chase ends.

In [ ]:
print("matches                 :", matches["match_id"].nunique())
print("innings rows            :", len(matches))
print("balls in deliveries file:", len(balls))
print("illegal deliveries      :", (balls["delivery_type"] != "legal").sum())
legal = balls[balls["delivery_type"] == "legal"]
print("avg balls per innings   :", round(len(legal) / legal.groupby(["match_id","innings"]).ngroups, 1))
matches.groupby("innings")[["runs", "wickets", "run_rate"]].mean()

## 2 · Match-level view: batting first vs chasing

In [ ]:
first_inn = matches[matches["innings"] == 1].set_index("match_id")
second_inn = matches[matches["innings"] == 2].set_index("match_id")
head_to_head = pd.DataFrame({
    "bat_first_runs": first_inn["runs"],
    "chase_runs": second_inn["runs"],
    "winner": first_inn["winner"],
    "chase_failed": (first_inn["winner"] == first_inn["batting_team"]),
})
print("batting first won %.0f%% of these matches"
      % (head_to_head["chase_failed"].mean() * 100))
head_to_head.head(8)

## 3 · Phase analysis — where the runs come from
T20 is really three different games: powerplay (1–6), middle (7–15), death (16–20).

In [ ]:
def phase(over):
    if over <= 6:
        return "Powerplay (1-6)"
    if over <= 15:
        return "Middle (7-15)"
    return "Death (16-20)"

balls["phase"] = balls["over"].map(phase)
phases = (balls.groupby("phase")
               .agg(balls=("total_runs", "size"),
                    runs=("total_runs", "sum"),
                    wickets=("is_wicket", "sum")))
phases["runs_per_over"] = phases["runs"] / phases["balls"] * 6
phases["balls_per_wicket"] = phases["balls"] / phases["wickets"]
phases["dot_pct"] = (balls[balls["total_runs"] == 0].groupby("phase").size()
                     / phases["balls"] * 100)

fig, ax = plt.subplots()
ax.bar(phases.index, phases["runs_per_over"], color=["#2563eb", "#f59e0b", "#ef4444"])
ax.set_ylabel("runs per over"); ax.set_title("Scoring rate by phase")
plt.tight_layout(); plt.show()
phases.round(2)

## 4 · Over-by-over scoring pattern

In [ ]:
per_over = balls.groupby("over")["total_runs"].mean().mul(6)
fig, ax = plt.subplots()
ax.plot(per_over.index, per_over.values, marker="o", color="#7c3aed", lw=2)
ax.axvspan(0.5, 6.5, color="#2563eb", alpha=.08)
ax.axvspan(15.5, 20.5, color="#ef4444", alpha=.08)
ax.set_xticks(range(1, 21))
ax.set_xlabel("over"); ax.set_ylabel("runs per over")
ax.set_title("The T20 shape: steady middle, steep finish")
plt.tight_layout(); plt.show()

## 5 · Batting leaderboard
Named aggregations keep the column names readable. Only batters with a
meaningful number of balls are ranked.

In [ ]:
batting = (balls.groupby("batter")
                .agg(innings=("match_id", "nunique"),
                     balls=("delivery_type", lambda s: (s == "legal").sum()),
                     runs=("batter_runs", "sum"),
                     fours=("batter_runs", lambda s: (s == 4).sum()),
                     sixes=("batter_runs", lambda s: (s == 6).sum())))
batting["strike_rate"] = batting["runs"] / batting["balls"] * 100
qualified = batting[batting["balls"] >= 60].sort_values("runs", ascending=False)
qualified.round(2)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(qualified["strike_rate"], qualified["runs"] / 10, s=70,
           color="#0ea5e9", alpha=.8)
for name, row in qualified.head(6).iterrows():
    ax.annotate(name, (row["strike_rate"], row["runs"] / 10), fontsize=9,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("strike rate"); ax.set_ylabel("runs × 10")
ax.set_title("Strike rate vs output (qualified batters)")
plt.tight_layout(); plt.show()

## 6 · Bowling leaderboard — normalise for overs bowled
Economy = runs conceded per 6 balls. Never rank bowlers on raw runs conceded.

In [ ]:
bowling = (balls.groupby("bowler")
                .agg(balls=("delivery_type", lambda s: (s == "legal").sum()),
                     runs=("total_runs", "sum"),
                     wickets=("is_wicket", "sum")))
bowling["overs"] = bowling["balls"] / 6
bowling["economy"] = bowling["runs"] / bowling["balls"] * 6
bowling["strike_rate_ball"] = np.where(bowling["wickets"] > 0,
                                       bowling["balls"] / bowling["wickets"], np.nan)
bowling[bowling["overs"] >= 4].sort_values("economy").round(2)

## 7 · Death-overs specialists

In [ ]:
death = balls[balls["over"] >= 16]
death_bowl = (death.groupby("bowler")
                   .agg(balls=("delivery_type", lambda s: (s == "legal").sum()),
                        runs=("total_runs", "sum"),
                        wickets=("is_wicket", "sum")))
death_bowl["economy"] = death_bowl["runs"] / death_bowl["balls"] * 6
death_bowl[death_bowl["balls"] >= 12].sort_values("economy").round(2)

## 8 · Insights
1. **Death overs cost roughly 2× the powerplay** — run rates climb from about
   8 to 11+ per over, so bowling plans must be phase specific, not batter specific.
2. **Dot balls are the hidden currency**: the middle phase produces the most
   dots, and pressure builds exactly there.
3. **Strike rate separates the top order** more than raw runs do — the highest
   scorers in this sample are not the fastest scorers.
4. **Economy leaderboards reshuffle completely** once you filter to 4+ overs;
   small-sample bowlers look brilliant and aren't.
5. **Batting first won the majority** of these matches, but margins are small —
   one more middle-over wicket is usually the difference.

### Extensions
- Add more matches by extending the data pull and re-run everything as-is.
- Build a batter-vs-bowler matrix with `pivot_table`.
- Reproduce this notebook in SQL: window functions for cumulative runs per over.